In [ ]:
# ============ IMPORTS ============
import pandas as pd
import numpy as np
import optuna
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from src.triple_barrier import triple_barrier
import optuna.visualization as vis
from scipy.stats import entropy as scipy_entropy

# ========== CONFIG ==========
window = 7  # days
min_label_entropy = 0.75 # Lest
# ============================

# ============ LOAD CACHED DATA ============
engineered_features_cache = pd.read_pickle('data_cache/engineered_features.pkl')
bitcoin_price_and_features = engineered_features_cache['bitcoin_price_and_features']
bitcoin_features = engineered_features_cache['bitcoin_features']

barrier_optimise_df = bitcoin_features.copy()
barrier_optimise_df['Close'] = bitcoin_price_and_features['Close']

# Use only the TRAINING portion (80%) for the Optuna search.
split_idx   = int(len(barrier_optimise_df) * 0.8)
barrier_optimise_df   = barrier_optimise_df.iloc[:split_idx]

feature_cols = [c for c in barrier_optimise_df.columns
                if c not in ['Close', 'Volatility_EWMA']]

# ============ OBJECTIVE FUNCTION ============
# This is the function Optuna calls once per trial.
# It receives a `trial` object, asks it for parameter suggestions,
# runs the code with those parameters, and returns a score.

def objective(trial):
    # Step 1: Optuna suggests parameter values
    profit_mult = trial.suggest_float("profit_mult", 0.5, 3, step=0.1)
    stop_mult   = trial.suggest_float("stop_mult",   0.5, 3, step=0.1)

    # Step 2: Run triple_barrier with those parameters
    triple_barrier_df = triple_barrier(
        price_series      = barrier_optimise_df['Close'],
        volatility_series = barrier_optimise_df['Volatility_EWMA'],
        holding_period    = window,          # keep consistent with your notebook
        profit_mult       = profit_mult,
        stop_mult         = stop_mult,
        min_ret_threshold = 0.005
    )

    # Step 3: Attach labels and drop NaN rows
    temp_df = barrier_optimise_df.copy()
    temp_df['label'] = triple_barrier_df['labels']
    temp_df = temp_df.dropna(subset=['label'])
    
    X = temp_df[feature_cols]
    y = temp_df['label'].astype(int)

    # Step 4: Guard against imbalanced label distributions
    # .reindex([-1, 0, 1], fill_value=0) makes sure counts always has all 3 classes
    # in all 3 runs, even if a class didn't appear. 
    # Missing class proportion (fill_value) = 0
    counts = y.value_counts(normalize=True).reindex([-1, 0, 1], fill_value=0)

    # scipy_entropy(counts) finds Shannon entropy: H = -sum(p * log(p))
    # Dividing by log(3) normalises H to the range [0, 1]:
    label_entropy = scipy_entropy(counts) / np.log(3)

    # These appear in study.trials_dataframe() and in MLflow automatically.
    # label_entropy: the balance score computed above
    trial.set_user_attr('label_entropy', round(float(label_entropy), 4))

    # counts.max() picks the highest value from [-1, 0, +1] proportions.
    trial.set_user_attr('lwr_barrier_pct', round(float(counts[-1]), 4))
    trial.set_user_attr('vert_barrier_pct', round(float(counts[0]),  4))
    trial.set_user_attr('upr_barrier_pct', round(float(counts[1]),  4))
    trial.set_user_attr('max_class_pct', round(float(counts.max()), 4))

    if label_entropy < min_label_entropy: # Give worst possible score to avoid
        return 0.0

    # Step 5: Time-series cross-validation
    cv_split = TimeSeriesSplit(n_splits=5,gap = window - 1) # 5 sequential train/test windows across the data.
        
    model  = RandomForestClassifier(
                n_estimators=100,
                max_depth=8,           # match your model_training.ipynb
                class_weight='balanced',
                random_state=42,
                n_jobs = -1
             )

    # cross_val_score trains and evaluates the model on each fold,
    # returns an array of 5 scores. We return the mean.
    scores = cross_val_score(model, X, y, cv=cv_split, scoring='f1_weighted')
    
    trial.set_user_attr('f1_std',  round(float(scores.std()),  4))
    for i, s in enumerate(scores):
        trial.set_user_attr(f'fold_{i}_f1', round(float(s), 4))
        
    return float(scores.mean())

In [ ]:
# ============ MLFLOW AND OPTUNA SET UP ============
mlflow.set_experiment("optuna_barrier_search")

# TPESampler uses Bayesian optimisation to focus trials on promising regions.
study = optuna.create_study(
    direction = "maximize",           # we want the HIGHEST f1_weighted
    sampler   = optuna.samplers.TPESampler(seed=42)  # continuous search, reproducible
)

In [ ]:
# ============ OPTUNA STUDY RUN ============
study.optimize(objective, n_trials = 50, n_jobs = -1)

best = study.best_params
print(f"Best profit_mult: {best['profit_mult']}")
print(f"Best stop_mult:   {best['stop_mult']}")
print(f"Best CV F1:       {study.best_value:.4f}")

In [ ]:
# ============ MLFLOW LOG ============
with mlflow.start_run(run_name="barrier_search_summary"):

    # Best parameters
    mlflow.log_params(study.best_trial.params)

    # Study-level metrics
    all_values = [t.value for t in study.trials if t.value is not None]
    mlflow.log_metric("best_cv_f1",      study.best_trial.value)
    mlflow.log_metric("mean_cv_f1",      float(np.mean(all_values)))
    mlflow.log_metric("std_cv_f1",       float(np.std(all_values)))
    mlflow.log_metric("n_trials",        len(study.trials))
    mlflow.log_metric("n_failed_trials", sum(1 for t in study.trials if t.value is None))

    # All the trials data
    optuna_trials_data = study.trials_dataframe().set_index('number')
    optuna_trials_data = pd.DataFrame(optuna_trials_data)
    optuna_trials_data.to_pickle("data_cache/optuna_trials_data.pkl")
    mlflow.log_artifact("data_cache/optuna_trials_data.pkl")
    
    # All visualisations as interactive HTML
    plots = {
        "optimisation_history": vis.plot_optimization_history(study),
        "contour":              vis.plot_contour(study),
        "slice":                vis.plot_slice(study),
        "param_importances":    vis.plot_param_importances(study),
        "parallel_coordinate":  vis.plot_parallel_coordinate(study),
        "rank":                 vis.plot_rank(study),
        "edf":                  vis.plot_edf(study),
        "timeline":             vis.plot_timeline(study),
    }
    for name, fig in plots.items():
        path = f"data_cache/optuna_{name}.html"
        fig.write_html(path)
        mlflow.log_artifact(path)